In [3]:
!mkdir -p src
!pip install -e . --no-build-isolation

Obtaining file:///workspace/auto_personality
  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 84.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 111.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 108.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.6 MB/s eta 0:00:00
  Building editable for personality-llm-pipeline (pyproject.toml) ... d

In [1]:
import subprocess
subprocess.run([
    "pip", "install",
    "transformers==4.45.0",
    "--quiet"
], check=True)
print("✅ 완료 — 커널 재시작 후 실행")


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


✅ 완료 — 커널 재시작 후 실행


In [2]:
# 셀 추가 — tokenizers 버전 고정
import subprocess
subprocess.run([
    "pip", "install", "tokenizers==0.20.3", "--quiet"
], check=True)
print("✅ 완료 — 커널 재시작 후 다시 실행")

✅ 완료 — 커널 재시작 후 다시 실행



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [4]:
import torch
import gc

def clear_vram():
    # 1. Python 레벨의 가비지 컬렉션 실행 (안 쓰이는 객체 정리)
    gc.collect()
    
    # 2. GPU 캐시 비우기
    torch.cuda.empty_cache()
    
    # (선택 사항) 만약 특정 변수를 확실히 지우고 싶다면:
    # del model
    # del optimizer
    # gc.collect()
    # torch.cuda.empty_cache()

# 실행
clear_vram()

import torch

# 1. 현재 할당된 메모리 (Active Memory)
allocated = torch.cuda.memory_allocated() / 1024**3
# 2. 캐시된 메모리 (Reserved Memory)
reserved = torch.cuda.memory_reserved() / 1024**3
# 3. 전체 용량 및 가용 용량 상세 확인
total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"현재 모델이 점유 중: {allocated:.2f} GB")
print(f"PyTorch가 예약한 캐시: {reserved:.2f} GB")
print(f"해당 GPU 전체 용량: {total_memory:.2f} GB")
print(f"실제 물리적 여유 공간: {total_memory - reserved:.2f} GB")

현재 모델이 점유 중: 0.00 GB
PyTorch가 예약한 캐시: 0.00 GB
해당 GPU 전체 용량: 79.25 GB
실제 물리적 여유 공간: 79.25 GB


In [5]:
import psutil
import os

def check_system_resource():
    # 1. CPU 사용량 확인 (전체 코어 평균 %)
    cpu_usage = psutil.cpu_percent(interval=1)
    
    # 2. RAM 사용량 확인
    mem = psutil.virtual_memory()
    total = mem.total / 1024**3
    available = mem.available / 1024**3
    percent = mem.percent
    
    # 3. 현재 파이썬 프로세스가 먹고 있는 RAM
    process = psutil.Process(os.getpid())
    process_mem = process.memory_info().rss / 1024**3

    print(f"--- 시스템 상태 ---")
    print(f"전체 CPU 사용률: {cpu_usage}%")
    print(f"RAM 전체 용량: {total:.2f} GB")
    print(f"RAM 사용 가능: {available:.2f} GB ({100-percent}% 남음)")
    print(f"현재 스크립트 점유: {process_mem:.2f} GB")

check_system_resource()

import torch
import gc
import os
import ctypes

def vacuum_cleaner():
    # 1. 모든 전역 변수 중 모델/텐서 관련 객체 삭제
    # (메모리를 많이 먹는 변수명을 직접 넣으세요)
    target_vars = ['model', 'tokenizer', 'optimizer', 'outputs', 'inputs']
    for var in target_vars:
        if var in globals():
            del globals()[var]
    
    # 2. 파이썬 가비지 컬렉터 가동
    gc.collect()
    
    # 3. GPU 메모리 비우기 (VRAM)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    
    # 4. C-라이브러리 수준에서 RAM 반환 (매우 중요)
    # 파이썬이 OS로부터 빌려온 뒤 들고 있는 메모리를 강제로 환수합니다.
    try:
        libc = ctypes.CDLL("libc.so.6")
        libc.malloc_trim(0)
    except:
        pass

    print("메모리 청소 완료!")

vacuum_cleaner()

--- 시스템 상태 ---
전체 CPU 사용률: 15.4%
RAM 전체 용량: 944.44 GB
RAM 사용 가능: 854.09 GB (90.4% 남음)
현재 스크립트 점유: 0.39 GB
메모리 청소 완료!


In [ ]:
import sys
sys.path.insert(0, "")

import run_pipeline
from run_pipeline import run_eval
from evaluator import _load_bfi_json          # ← 이걸로 바꿈
from merger import DynamicMerger
from openai import OpenAI
import numpy as np
import json
from pathlib import Path

bfi_meta      = _load_bfi_json()              # ← BFI_META 대신 이렇게
openai_client = OpenAI(api_key=run_pipeline.OPENAI_API_KEY)
print("✅ import 완료")

ImportError: cannot import name 'Sentinel' from 'typing_extensions' (/usr/local/lib/python3.11/dist-packages/typing_extensions.py)

In [ ]:
RESULT_DIR = ""
N_REPEAT   = 5
BIG_FIVE   = ["OPN", "CON", "EXT", "AGR", "NEU"]
BFI_KEY_MAP = {
    "OPN": "openness",
    "CON": "conscientiousness",
    "EXT": "extraversion",
    "AGR": "agreeableness",
    "NEU": "neuroticism",
}
MODELS = {
    "llama": {
        "base_model_path": "",
        "phi_dir":         "",
    },
    "qwen": {
        "base_model_path": "",
        "phi_dir":         "",
    },
}
print("✅ 설정 완료")

In [ ]:
import gc
import torch

def get_base_bfi(merger_obj):
    bfi, _, _, _ = run_eval(
        model=merger_obj.base_model,
        tokenizer=merger_obj.tokenizer,
        bfi_meta=bfi_meta,                    # ← 여기
        openai_client=openai_client,
    )
    print(f"[Base BFI] {bfi}")
    return bfi

def measure_single_phi(dim, merger_obj, base_bfi):
    alpha = {d: 0.0 for d in BIG_FIVE}
    alpha[dim] = 1.0
    #-0.1 low 가 됨 바꿔서 시험하면 됨
    delta_list = {k: [] for k in BFI_KEY_MAP.values()}

    for rep in range(1, N_REPEAT + 1):
        print(f"  [{dim}] {rep}/{N_REPEAT}")

        import gc, torch
        gc.collect()
        torch.cuda.empty_cache()
        print(f"  VRAM 여유: {torch.cuda.mem_get_info()[0]/1e9:.1f}GB")

        merged = merger_obj.merge(alpha)
        bfi, _, _, _ = run_eval(
            model=merged,
            tokenizer=merger_obj.tokenizer,
            bfi_meta=bfi_meta,
            openai_client=openai_client,
        )
        for k in BFI_KEY_MAP.values():
            delta_list[k].append(bfi.get(k, 0) - base_bfi.get(k, 0))

        # _reset_to_base 제거 (merge()가 자동으로 base 복원함)
        gc.collect()
        torch.cuda.empty_cache()
        print(f"  [{dim}] BFI={bfi}")

    avg = {k: round(float(np.mean(v)), 4) for k, v in delta_list.items()}
    print(f"[{dim}] 평균 변화량={avg}")
    return avg
def build_and_save(results, model_name):
    C = np.zeros((5, 5))
    for i, phi in enumerate(BIG_FIVE):
        for j, bfi in enumerate(BIG_FIVE):
            C[i][j] = results[phi].get(BFI_KEY_MAP[bfi], 0.0)

    out_dir = Path(RESULT_DIR) / model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "C_matrix.npy", C)
    with open(out_dir / "C_matrix.json", "w") as f:
        json.dump({"model": model_name, "C": C.tolist(), "raw": results}, f, indent=2)

    print(f"\nC Matrix [{model_name}]")
    print(f"{'':8}", *[f"{d:>8}" for d in BIG_FIVE])
    for i, phi in enumerate(BIG_FIVE):
        row = f"φ_{phi:4}  "
        for j in range(5):
            row += f"{C[i][j]:>7.3f}{'★' if i==j else ' '}"
        print(row)
    print(f"✅ {out_dir}")
    return C

In [ ]:
# HF 로그인 후 온라인에서 template 가져오기
from huggingface_hub import login
login(token=run_pipeline.HF_TOKEN)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

# template만 추출해서 로컬에 저장
import json
with open("") as f:
    cfg = json.load(f)

cfg["chat_template"] = tokenizer.chat_template

with open("", "w") as f:
    json.dump(cfg, f, indent=2)

print("✅ chat_template 저장 완료")
print(tokenizer.chat_template[:100])

In [ ]:
# ══ 재초기화 ══
import sys, json, gc
import numpy as np
import torch
from pathlib import Path

sys.path.insert(0, "")

import run_pipeline
from run_pipeline import run_eval
from evaluator import _load_bfi_json
from merger import DynamicMerger
from openai import OpenAI

BIG_FIVE    = ["OPN", "CON", "EXT", "AGR", "NEU"]
BFI_KEY_MAP = {
    "OPN": "openness",
    "CON": "conscientiousness",
    "EXT": "extraversion",
    "AGR": "agreeableness",
    "NEU": "neuroticism",
}
RESULT_DIR    = ""
bfi_meta      = _load_bfi_json()
openai_client = OpenAI(api_key=run_pipeline.OPENAI_API_KEY)

# C 행렬 + raw 결과 로드
with open(f"{RESULT_DIR}/llama/C_matrix.json") as f:
    data_llama = json.load(f)
C_llama       = np.array(data_llama["C"])
results_llama = data_llama["raw"]

base_bfi_llama = {
    'extraversion': 3.75, 'neuroticism': 2.5,
    'conscientiousness': 3.111, 'agreeableness': 4.111,
    'openness': 5.0
}

# llama merger 로드 (phi 로더 역할)
merger_llama = DynamicMerger(
    base_model_path="/workspace/auto_personality/llama_base",
    phi_dir="/workspace/auto_personality/phi_vectors_llama",
    method="task_arithmetic",
    dare_drop_rate=0.5, dare_rescale=True,
    dare_strategy="random", ties_trim_rate=0.7,
    scaling_coefficient=1.0,
)
print("✅ 재초기화 완료 — 직교화 셀 실행 가능!")

10:17:48 [INFO] [Merger] 사용 디바이스: cuda
10:17:48 [INFO] [Merger] base 모델 로드 (GPU): /workspace/auto_personality/llama_base
10:17:53 [INFO] [Merger] θ_base 복사본 → GPU 보관
10:17:53 [INFO] [Merger] θ_base GPU 보관 완료
10:17:53 [INFO] [Merger] φ 벡터 캐싱 시작 (10개 → CPU RAM)
10:18:25 [INFO]   [φ 캐시] OPN_High ✅
10:18:54 [INFO]   [φ 캐시] CON_High ✅
10:19:21 [INFO]   [φ 캐시] EXT_High ✅
10:19:50 [INFO]   [φ 캐시] AGR_High ✅
10:20:18 [INFO]   [φ 캐시] NEU_High ✅
10:20:18 [INFO] [Merger] φ 캐시 완료 — 5개 CPU RAM 상주
10:20:18 [INFO] [Merger] 초기화 완료 — method=task_arithmetic


✅ 재초기화 완료 — 직교화 셀 실행 가능!


In [ ]:


import gc
import json
import logging
import os
import time
import ctypes

import torch

# ──────────────────────────────────────────────────────────────────────────────
# 시스템 설정
# ──────────────────────────────────────────────────────────────────────────────
torch.set_num_threads(1)

try:
    libc = ctypes.CDLL("libc.so.6")
except Exception:
    libc = None

# ──────────────────────────────────────────────────────────────────────────────
# ★★★ 여기만 수정하세요 ★★★
# ──────────────────────────────────────────────────────────────────────────────

PHI_DIR    = ""
OUTPUT_DIR = "./ortho_phi_vectors"

# 처리 순서 (첫 번째가 기준축 — 손실 없음)
KEYS = ["CON_High", "NEU_High", "EXT_High", "AGR_High", "OPN_High"]

# ──────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


# ==============================================================================
# 유틸
# ==============================================================================

def os_free():
    """gc + CUDA cache + OS 레벨 RAM 즉시 반환"""
    gc.collect()
    torch.cuda.empty_cache()
    if libc:
        libc.malloc_trim(0)


def dict_to_flat(param_dict: dict) -> torch.Tensor:
    """
    param dict → 1D bfloat16 CPU tensor

    ★ pop으로 파괴하면서 변환 → 변환 중 RAM 스파이크 방지
    변환 완료 후 param_dict는 빈 dict가 됨 (별도 free 불필요)
    """
    total_numel = sum(v.numel() for v in param_dict.values())
    flat = torch.empty(total_numel, dtype=torch.bfloat16, device="cpu")

    offset = 0
    for k in sorted(list(param_dict.keys())):
        v = param_dict.pop(k)
        numel = v.numel()
        flat[offset:offset + numel] = v.cpu().bfloat16().reshape(-1)
        offset += numel
        del v

    return flat


def phi_path(key: str) -> str:
    return os.path.join(PHI_DIR, f"phi_{key}.pt")


def load_phi(key: str) -> dict:
    path = phi_path(key)
    assert os.path.exists(path), f"phi 파일 없음: {path}"
    return torch.load(path, map_location="cpu", weights_only=True)


def ram_usage() -> str:
    """현재 프로세스 RAM 사용량 (psutil 있을 때만)"""
    try:
        import psutil
        proc = psutil.Process(os.getpid())
        rss = proc.memory_info().rss / (1024 ** 3)
        vm  = psutil.virtual_memory()
        return f"프로세스 {rss:.1f}GB | 시스템 여유 {vm.available/1024**3:.1f}GB"
    except ImportError:
        return "(psutil 없음)"


# ==============================================================================
# Gram-Schmidt 직교화
# ==============================================================================

def gram_schmidt_step(key: str, prev_keys: list, gs_dir: str) -> str | None:
    """
    phi_{key}.pt 로드 → 이전 직교 벡터들의 투영 제거 → 저장

    [ 동시 RAM 사용량 ]
    v_i 처리 중    : v_i(15GB) + u_j(15GB) = 30GB
    저장 직전      : result(15GB) — v_i는 이미 해제
    저장 완료 후   : 0GB
    """
    # ── 1. phi 로드 → flat 변환 (pop으로 파괴) ─────────────────
    phi_dict = load_phi(key)
    ref_shapes = {k: (v.shape, v.dtype) for k, v in phi_dict.items()}
    v_i = dict_to_flat(phi_dict)
    # phi_dict는 pop으로 이미 비어있음 → 별도 해제 불필요
    del phi_dict
    os_free()

    logger.info(f"  [{key}] GS 시작 | norm: {v_i.norm():.6f} | {ram_usage()}")

    # ── 2. 이전 직교 벡터들의 투영 제거 ───────────────────────
    for prev_key in prev_keys:
        prev_path = os.path.join(gs_dir, f"ortho_gs_{prev_key}.pt")
        if not os.path.exists(prev_path):
            logger.warning(f"  [{key}] 이전 파일 없음: {prev_path}")
            continue

        # 이전 벡터 로드 → flat 변환 (pop으로 파괴)
        prev_dict = torch.load(prev_path, map_location="cpu", weights_only=True)
        u_j = dict_to_flat(prev_dict)
        del prev_dict
        # prev_dict는 pop으로 이미 비어있음

        dot_uu = torch.dot(u_j, u_j).item()
        if dot_uu < 1e-12:
            del u_j
            os_free()
            continue

        # in-place 뺄셈 (새 텐서 생성 없음)
        scalar = torch.dot(v_i, u_j).item() / dot_uu
        v_i.sub_(u_j, alpha=scalar)

        # u_j 즉시 해제 + OS 반환
        del u_j
        os_free()
        logger.info(f"  [{key}] -{prev_key} 투영 제거 | scalar={scalar:.6f} | {ram_usage()}")

    new_norm = v_i.norm().item()
    logger.info(f"  [{key}] GS 완료 | norm: {new_norm:.6f}")

    if new_norm < 1e-8:
        logger.warning(f"  [{key}] ⚠️  영벡터! 저장 건너뜀")
        del v_i
        os_free()
        return None

    # ── 3. result dict 복원 ────────────────────────────────────
    # v_i 슬라이스 → clone으로 v_i와 연결 끊기
    result = {}
    offset = 0
    for k in sorted(ref_shapes.keys()):
        shape, dtype = ref_shapes[k]
        numel = 1
        for s in shape:
            numel *= s
        result[k] = v_i[offset:offset + numel].reshape(shape).to(dtype).clone()
        offset += numel

    # ★ result 완성 후 v_i 즉시 해제 (30GB → 15GB)
    del v_i
    os_free()
    logger.info(f"  [{key}] v_i 해제 후 | {ram_usage()}")

    # ── 4. 저장 → result 즉시 해제 ────────────────────────────
    save_path = os.path.join(gs_dir, f"ortho_gs_{key}.pt")
    torch.save(result, save_path)
    del result
    os_free()

    size_mb = os.path.getsize(save_path) / (1024 ** 2)
    logger.info(f"  [{key}] 저장 완료 → {save_path} ({size_mb:.1f} MB) | {ram_usage()}")
    return save_path


# ==============================================================================
# 메인
# ==============================================================================

def run():
    gs_dir = os.path.join(OUTPUT_DIR, "gs_con")
    os.makedirs(gs_dir, exist_ok=True)

    logger.info("=" * 55)
    logger.info("  Gram-Schmidt 직교화 파이프라인")
    logger.info(f"  phi 경로  : {PHI_DIR}")
    logger.info(f"  출력 경로 : {gs_dir}")
    logger.info(f"  처리 순서 : {' → '.join(KEYS)}")
    logger.info(f"  메모리 전략: 최대 동시 사용 ~30GB")
    logger.info("=" * 55)

    # phi 파일 존재 확인
    missing = [k for k in KEYS if not os.path.exists(phi_path(k))]
    if missing:
        logger.error(f"phi 파일 없음: {missing}")
        return None

    start = time.time()
    manifest = {
        "phi_dir":    PHI_DIR,
        "gs_order":   KEYS,
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "gs_files":   {},
    }

    gs_saved_keys = []
    for i, key in enumerate(KEYS):
        logger.info(f"\n[{i+1}/{len(KEYS)}] ══ {key} ══")
        logger.info(f"  시작 전 RAM: {ram_usage()}")

        gs_path = gram_schmidt_step(key, gs_saved_keys, gs_dir)
        if gs_path is None:
            continue

        gs_saved_keys.append(key)
        manifest["gs_files"][key] = gs_path

        # 루프 종료 시점 OS 레벨 최종 청소
        os_free()
        logger.info(f"  [{key}] 완료 ✅ | {ram_usage()}")

    manifest["elapsed_sec"] = round(time.time() - start, 1)
    manifest_path = os.path.join(OUTPUT_DIR, "manifest_gs_open.json")
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    logger.info(f"\n{'='*55}")
    logger.info(f"  완료! 총 소요: {manifest['elapsed_sec']}초")
    logger.info(f"  저장: {len(manifest['gs_files'])}개 → {gs_dir}/")
    logger.info(f"{'='*55}")
    return manifest


if __name__ == "__main__":
    run()

10:44:50 [INFO] =======================================================
10:44:50 [INFO]   Gram-Schmidt 직교화 파이프라인
10:44:50 [INFO]   phi 경로  : /workspace/auto_personality/phi_vectors_llama
10:44:50 [INFO]   출력 경로 : ./ortho_phi_vectors/gs_con
10:44:50 [INFO]   처리 순서 : CON_High → NEU_High → EXT_High → AGR_High → OPN_High
10:44:50 [INFO]   메모리 전략: 최대 동시 사용 ~30GB
10:44:50 [INFO] =======================================================
10:44:50 [INFO] 
[1/5] ══ CON_High ══
10:44:50 [INFO]   시작 전 RAM: 프로세스 0.4GB | 시스템 여유 907.1GB
10:45:31 [INFO]   [CON_High] GS 시작 | norm: 2.000000 | 프로세스 15.4GB | 시스템 여유 892.0GB
10:45:33 [INFO]   [CON_High] GS 완료 | norm: 2.000000
10:45:42 [INFO]   [CON_High] v_i 해제 후 | 프로세스 15.4GB | 시스템 여유 892.0GB
10:46:10 [INFO]   [CON_High] 저장 완료 → ./ortho_phi_vectors/gs_con/ortho_gs_CON_High.pt (15316.6 MB) | 프로세스 0.4GB | 시스템 여유 906.9GB
10:46:10 [INFO]   [CON_High] 완료 ✅ | 프로세스 0.4GB | 시스템 여유 906.9GB
10:46:10 [INFO] 
[2/5] ══ NEU_High ══
10:46:10 [INFO]   시작 전 RAM: 프로세스 0.4GB |

In [ ]:
import importlib
import merger
importlib.reload(merger)  # ← 항상 원본으로 리셋

merger.MANIFEST = {
    "OPN": {"High": "phi_OPN_High.pt"},
    "CON": {"High": "phi_CON_High.pt"},
    "EXT": {"High": "phi_EXT_High.pt"},
    "AGR": {"High": "phi_AGR_High.pt"},
    "NEU": {"High": "phi_NEU_High.pt"},
}

# reload 직후에 원본 함수 저장 → 항상 진짜 원본
_true_original = merger.DynamicMerger._load_phi

def patched_load_phi(self, dim, direction):
    return _true_original(self, dim, "High")

merger.DynamicMerger._load_phi = patched_load_phi

merger_llama_lowdin = merger.DynamicMerger(
    base_model_path="",
    phi_dir="",
    method="task_arithmetic",
    dare_drop_rate=0.5, dare_rescale=True,
    dare_strategy="random", ties_trim_rate=0.7, scaling_coefficient=1.0,
)
print("✅ lowdin merger 로드 완료:", len(merger_llama_lowdin.phi_cache), "개")

00:30:02 [INFO] [Merger] 사용 디바이스: cuda
00:30:02 [INFO] [Merger] base 모델 로드 (GPU): /workspace/auto_personality/llama_base
00:30:29 [INFO] [Merger] θ_base 복사본 → GPU 보관
00:30:29 [INFO] [Merger] θ_base GPU 보관 완료
00:30:29 [INFO] [Merger] φ 벡터 캐싱 시작 (10개 → CPU RAM)
00:30:56 [INFO]   [φ 캐시] OPN_High ✅
00:31:20 [INFO]   [φ 캐시] CON_High ✅
00:31:43 [INFO]   [φ 캐시] EXT_High ✅
00:32:13 [INFO]   [φ 캐시] AGR_High ✅
00:32:41 [INFO]   [φ 캐시] NEU_High ✅
00:32:41 [INFO] [Merger] φ 캐시 완료 — 5개 CPU RAM 상주
00:32:41 [INFO] [Merger] 초기화 완료 — method=task_arithmetic


✅ lowdin merger 로드 완료: 5 개


In [11]:
import importlib
import evaluator  # 수정한 파일(모듈) 이름
import utils_interview
from openai import OpenAI
# 파일을 고친 후 이 줄을 실행하면 최신 코드가 반영됩니다.
importlib.reload(evaluator)
importlib.reload(utils_interview)

<module 'utils_interview' from '/workspace/auto_personality/utils_interview.py'>

In [ ]:
from openai import OpenAI
# 현재 client의 api_key 확인
print(openai_client.api_key[:20], "...")

# 새로 만들기
from openai import OpenAI
openai_client = OpenAI(api_key="")
print("✅ client 재생성 완료")

sk-proj-reEO4MUIiXOg ...
✅ client 재생성 완료


In [13]:
# ── 설정 ──
N_REPEAT = 5  # 시간 절약용 (논문은 3회 평균)

# lowdin 결과 측정이면
base_bfi_lowdin = get_base_bfi(merger_llama_lowdin)

results_lowdin = {}
for dim in BIG_FIVE:
    alpha = {d: 0.0 for d in BIG_FIVE}
    alpha[dim] = 1.0

    delta_list = {k: [] for k in BFI_KEY_MAP.values()}
    for rep in range(1, N_REPEAT + 1):
        print(f"  [{dim}] {rep}/{N_REPEAT}")
        gc.collect()
        torch.cuda.empty_cache()

        merged = merger_llama_lowdin.merge(alpha)  # ← lowdin!
        bfi, _, _, _ = run_eval(
            model=merged,
            tokenizer=merger_llama_lowdin.tokenizer,
            bfi_meta=bfi_meta,
            openai_client=openai_client,
        )
        for k in BFI_KEY_MAP.values():
            delta_list[k].append(bfi.get(k, 0) - base_bfi_lowdin.get(k, 0))

        gc.collect()
        torch.cuda.empty_cache()
        print(f"  [{dim}] BFI={bfi}")

    avg = {k: round(float(np.mean(v)), 4) for k, v in delta_list.items()}
    print(f"[{dim}] 평균 변화량={avg}")
    results_lowdin[dim] = avg

C_lowdin = build_and_save(results_lowdin, "llama_ortho_awd")

00:32:47 [INFO]   [Eval] BFI 44문항 수집 중...


[get_model_response] model_type=llama  |  device=cuda:0


Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device

00:34:23 [INFO]   [Eval] GPT-4o 채점 중...
 80%|████████  | 4/5 [00:44<00:10, 10.66s/it]

...

  [NEU] BFI={'extraversion': 2.25, 'neuroticism': 4.125, 'conscientiousness': 2.556, 'agreeableness': 2.333, 'openness': 3.3, 'emotional_stability': 1.875}
  [NEU] 5/5


01:17:31 [INFO]   [NEU] α=+1.0000 (High) ✅
01:17:32 [INFO] [Merger] 머징 완료 ✅
01:17:32 [INFO]   [Eval] BFI 44문항 수집 중...


[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device=cuda:0
[get_model_response] model_type=llama  |  device

01:18:29 [INFO]   [Eval] GPT-4o 채점 중...
100%|██████████| 5/5 [01:00<00:00, 12.03s/it]
01:19:30 [INFO]   [Eval] BFI: {'extraversion': 2.0, 'neuroticism': 4.375, 'conscientiousness': 2.444, 'agreeableness': 2.444, 'openness': 3.1, 'emotional_stability': 1.625}
01:19:30 [INFO]   [Eval] error_counts: {'is_multilanguage': 0, 'not_into_character': 0, 'contain_repeation': 0, 'is_multiround': 0}


  [NEU] BFI={'extraversion': 2.0, 'neuroticism': 4.375, 'conscientiousness': 2.444, 'agreeableness': 2.444, 'openness': 3.1, 'emotional_stability': 1.625}
[NEU] 평균 변화량={'openness': -1.42, 'conscientiousness': -1.5002, 'extraversion': -1.175, 'agreeableness': -1.4558, 'neuroticism': 2.138}

C Matrix [llama_ortho_awd]
              OPN      CON      EXT      AGR      NEU
φ_OPN     0.280★ -0.978   0.425   0.056   0.176 
φ_CON    -0.520   0.533★ -0.400   0.089   0.163 
φ_EXT     0.200  -0.133   1.087★  0.411  -0.124 
φ_AGR    -0.340   0.166  -0.138   0.389★  0.163 
φ_NEU    -1.420  -1.500  -1.175  -1.456   2.138★
✅ /workspace/auto_personality/caas/llama_ortho_awd


In [ ]:
import json
import numpy as np

def pearsonr_np(x, y):
    """scipy 없이 numpy로만 Pearson r 계산"""
    x, y = np.array(x), np.array(y)
    xm, ym = x - x.mean(), y - y.mean()
    r = np.dot(xm, ym) / (np.sqrt(np.dot(xm, xm)) * np.sqrt(np.dot(ym, ym)) + 1e-12)
    return float(r)

# ── 데이터 로드 ──
with open(f"{RESULT_DIR}/llama_task/C_matrix.json") as f:
    data_orig = json.load(f)
with open(f"") as f:
    data_ortho = json.load(f)

C_orig  = np.array(data_orig["C"])
C_ortho = np.array(data_ortho["C"])

# ── 1. C 행렬 자체 출력 ──
print("=" * 60)
print("[ 원본 phi C 행렬 ]")
print(f"{'':8}", *[f"{d:>8}" for d in BIG_FIVE])
for i, phi in enumerate(BIG_FIVE):
    row = f"φ_{phi:4}  "
    for j in range(5):
        star = "★" if i == j else " "
        row += f"{C_orig[i][j]:>7.3f}{star}"
    print(row)

print()
print("[ 직교화(GS) phi C 행렬 ]")
print(f"{'':8}", *[f"{d:>8}" for d in BIG_FIVE])
for i, phi in enumerate(BIG_FIVE):
    row = f"φ_{phi:4}  "
    for j in range(5):
        star = "★" if i == j else " "
        row += f"{C_ortho[i][j]:>7.3f}{star}"
    print(row)

# ── 2. 대각선 vs 비대각선 비교 ──
print("\n" + "=" * 60)
print("[ 타겟 효과 vs 간섭 효과 비교 ]")
print(f"{'':6} | {'원본':^20} | {'직교화':^20} | 개선")
print(f"{'Trait':6} | {'타겟':>8} {'간섭':>8} {'비율':>4} | {'타겟':>8} {'간섭':>8} {'비율':>4} | ")
print("-" * 65)

for i, dim in enumerate(BIG_FIVE):
    # 원본
    diag_o = C_orig[i, i]
    off_o  = np.mean([C_orig[i, j] for j in range(5) if j != i])
    ratio_o = abs(diag_o) / (abs(off_o) + 1e-8)

    # 직교화
    diag_q = C_ortho[i, i]
    off_q  = np.mean([C_ortho[i, j] for j in range(5) if j != i])
    ratio_q = abs(diag_q) / (abs(off_q) + 1e-8)

    better = "✅" if ratio_q > ratio_o else "❌"
    print(f"{dim:6} | {diag_o:>8.3f} {off_o:>8.3f} {ratio_o:>4.1f} | "
          f"{diag_q:>8.3f} {off_q:>8.3f} {ratio_q:>4.1f} | {better}")

# ── 3. Pearson 상관계수 비교 (논문 Table 9 형식) ──
print("\n" + "=" * 60)
print("[ Pearson 상관계수 비교 (논문 Table 9 형식) ]")
print(f"{'':12} | {'OPN':>6} {'CON':>6} {'EXT':>6} {'AGR':>6} {'NEU':>6} {'AVG':>6}")
print("-" * 55)

# alpha 스케일 벡터 (0~4 → 각 phi 인덱스)
scale_vec = np.arange(5)  # [0,1,2,3,4] = phi 인덱스

for label, C in [("원본 phi", C_orig), ("직교화(GS)", C_ortho)]:
    rs = []
    row = f"{label:12} |"
    for j in range(5):
        col = C[:, j]
        r = pearsonr_np(scale_vec, col)  # ← r, _ 아니고 r만!
        rs.append(r)
        row += f" {r:>6.3f}"
    avg_r = np.mean(rs)
    row += f" {avg_r:>6.3f}"
    print(row)
# ── 4. 간섭 총량 비교 ──
print("\n" + "=" * 60)
off_diag_orig  = np.mean(np.abs(C_orig  - np.diag(np.diag(C_orig))))
off_diag_ortho = np.mean(np.abs(C_ortho - np.diag(np.diag(C_ortho))))
reduction = (off_diag_orig - off_diag_ortho) / off_diag_orig * 100

print(f"[ 평균 간섭량 ]")
print(f"  원본 phi : {off_diag_orig:.4f}")
print(f"  직교화   : {off_diag_ortho:.4f}")
print(f"  간섭 감소: {reduction:+.1f}%  {'✅ 직교화 효과 있음!' if reduction > 0 else '❌ 개선 없음'}")


# 논문 방식: 각 phi의 타겟 trait 변화량 vs alpha 스케일
# C 행렬의 대각선 = alpha=1.0 일 때 변화량
# 비교 대상: [원본 C 대각선] vs [직교화 C 대각선]

print("\n[ 핵심 비교: 타겟 trait 변화량 (대각선) ]")
print(f"{'Trait':6} | {'원본':>8} | {'직교화':>8} | {'차이':>8}")
print("-" * 40)
total_orig = 0
total_ortho = 0
for i, dim in enumerate(BIG_FIVE):
    orig  = C_orig[i, i]
    ortho = C_ortho[i, i]
    diff  = ortho - orig
    better = "✅" if abs(ortho) >= abs(orig) else "❌"
    print(f"{dim:6} | {orig:>8.3f} | {ortho:>8.3f} | {diff:>+8.3f} {better}")
    total_orig  += abs(orig)
    total_ortho += abs(ortho)

print(f"\n{'합계':6} | {total_orig:>8.3f} | {total_ortho:>8.3f}")
print(f"\n[ 결론 ]")
print(f"  타겟 효과 총합: 원본={total_orig:.3f} → 직교화={total_ortho:.3f}")
print(f"  간섭 총량:      원본={off_diag_orig:.4f} → 직교화={off_diag_ortho:.4f}")
print(f"  Signal/Noise:  원본={total_orig/off_diag_orig:.2f} → 직교화={total_ortho/off_diag_ortho:.2f}")

[ 원본 phi C 행렬 ]
              OPN      CON      EXT      AGR      NEU
φ_OPN    -0.100★  0.222   0.075   0.156   0.050 
φ_CON    -0.760   1.622★ -0.925   0.111  -0.125 
φ_EXT    -0.080   0.800   0.950★  0.489  -0.125 
φ_AGR    -0.680   0.933  -0.550   0.333★ -0.250 
φ_NEU    -1.780  -0.622  -1.650  -1.600   1.500★

[ 직교화(GS) phi C 행렬 ]
              OPN      CON      EXT      AGR      NEU
φ_OPN     0.460★ -0.811  -0.250   0.156  -0.050 
φ_CON    -0.020   0.189★ -0.537   0.000   0.050 
φ_EXT     0.400  -0.289   0.562★  0.100  -0.350 
φ_AGR     0.100  -0.155  -0.550   0.244★ -0.200 
φ_NEU    -0.080  -0.967  -0.912  -0.322   1.062★

[ 타겟 효과 vs 간섭 효과 비교 ]
       |          원본          |         직교화          | 개선
Trait  |       타겟       간섭   비율 |       타겟       간섭   비율 | 
-----------------------------------------------------------------
OPN    |   -0.100    0.126  0.8 |    0.460   -0.239  1.9 | ✅
CON    |    1.622   -0.425  3.8 |    0.189   -0.127  1.5 | ❌
EXT    |    0.950    0.271  3.5 |  

In [21]:
import psutil
vm = psutil.virtual_memory()
print(f"전체: {vm.total/1024**3:.1f} GB")
print(f"여유: {vm.available/1024**3:.1f} GB")

전체: 944.4 GB
여유: 824.4 GB


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
run_lowdin_ortho.py

Löwdin (Symmetric Orthogonalization) 방식으로
personality phi 벡터들의 상관관계를 제거한다.

[ 속도 개선 핵심 ]
  기존: 청크마다 5개 phi 파일을 반복 로드 → 8000번 디스크 읽기 💀
  개선: phi 5개를 한 번만 로드 → 청크는 메모리 내에서 처리 ✅

[ 메모리 사용량 ]
  phi 5개 동시: 15GB × 5 = 75GB (RAM 117GB 중 75GB 사용)
  청크 처리 시: V_chunk + U_chunk 수백 MB 추가
  → 총 ~76GB, 117GB 내에서 안전

[ 처리 흐름 ]
  STEP 1: phi 5개 한 번에 로드 (75GB)
  STEP 2: 청크 단위로 S = V^T V 누적 계산 (5×5)
  STEP 3: S^{-1/2} 계산 (5×5, 무시 수준)
  STEP 4: 청크 단위로 U = V S^{-1/2} 계산 → 임시 파일 저장 → 해제
  STEP 5: phi 5개 해제
  STEP 6: 임시 파일 합쳐서 최종 phi_*.pt 저장
  STEP 7: 직교성 검증

[ Jupyter 사용법 ]
  import run_lowdin_ortho as low
  low.PHI_DIR    = "/workspace/auto_personality/phi_vectors_llama"
  low.OUTPUT_DIR = "./ortho_phi_vectors"
  low.KEYS       = ["CON_High", "NEU_High", "EXT_High", "AGR_High", "OPN_High"]
  low.run()

[ 저장 결과 ]
  ortho_phi_vectors/lowdin/
    phi_CON_High.pt  ← 직교화된 벡터 (merger.py 호환 파일명)
    phi_NEU_High.pt
    phi_EXT_High.pt
    phi_AGR_High.pt
    phi_OPN_High.pt
"""

import ctypes
import gc
import json
import logging
import os
import time

import torch

# ──────────────────────────────────────────────────────────────────────────────
# 시스템 설정
# ──────────────────────────────────────────────────────────────────────────────
torch.set_num_threads(4)

try:
    libc = ctypes.CDLL("libc.so.6")
except Exception:
    libc = None

# ──────────────────────────────────────────────────────────────────────────────
# ★★★ 여기만 수정하세요 ★★★
# ──────────────────────────────────────────────────────────────────────────────

PHI_DIR    = ""
OUTPUT_DIR = "./ortho_phi_vectors"

KEYS = ["CON_High", "NEU_High", "EXT_High", "AGR_High", "OPN_High"]

# 청크 크기: 메모리 내 처리용 (파일 I/O와 무관)
# V_chunk = chunk_size × 5 × 4bytes(float32) = 10M × 5 × 4 = 200MB
CHUNK_SIZE = 10_000_000  # 10M 파라미터

EPS = 1e-12

# ──────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


# ==============================================================================
# 유틸
# ==============================================================================

def os_free():
    gc.collect()
    torch.cuda.empty_cache()
    if libc:
        libc.malloc_trim(0)


def phi_path(key: str) -> str:
    return os.path.join(PHI_DIR, f"phi_{key}.pt")


def load_phi(key: str) -> dict:
    path = phi_path(key)
    assert os.path.exists(path), f"phi 파일 없음: {path}"
    return torch.load(path, map_location="cpu", weights_only=True)


def ram_usage() -> str:
    try:
        import psutil
        proc = psutil.Process(os.getpid())
        rss  = proc.memory_info().rss / (1024 ** 3)
        vm   = psutil.virtual_memory()
        return f"RSS {rss:.1f}GB | 여유 {vm.available/1024**3:.1f}GB"
    except ImportError:
        return ""


# ==============================================================================
# Löwdin 메인
# ==============================================================================

def run():
    out_dir = os.path.join(OUTPUT_DIR, "lowdin")
    tmp_dir = os.path.join(out_dir, "_tmp")
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(tmp_dir, exist_ok=True)

    n = len(KEYS)

    logger.info("=" * 60)
    logger.info("  Löwdin (Symmetric Orthogonalization)")
    logger.info(f"  phi 폴더  : {PHI_DIR}")
    logger.info(f"  출력 폴더 : {out_dir}")
    logger.info(f"  대상      : {' | '.join(KEYS)}")
    logger.info("=" * 60)

    missing = [k for k in KEYS if not os.path.exists(phi_path(k))]
    if missing:
        logger.error(f"phi 파일 없음: {missing}")
        return None

    start = time.time()

    # ── STEP 1: phi 5개 한 번에 로드 ──────────────────────────
    logger.info(f"\nSTEP 1: phi {n}개 로드 중... (총 ~{n*15}GB 예상)")
    phis = {}
    for key in KEYS:
        logger.info(f"  [{key}] 로드 중...")
        phis[key] = load_phi(key)
        logger.info(f"  [{key}] 완료 | {ram_usage()}")

    # 파라미터 키 목록 + 청크 경계 계산
    sorted_param_keys = sorted(phis[KEYS[0]].keys())
    total_params = sum(phis[KEYS[0]][k].numel() for k in sorted_param_keys)
    logger.info(f"\n  총 파라미터: {total_params:,}")

    chunks, cur, cur_size = [], [], 0
    for pk in sorted_param_keys:
        numel = phis[KEYS[0]][pk].numel()
        cur.append((pk, numel))
        cur_size += numel
        if cur_size >= CHUNK_SIZE:
            chunks.append(cur)
            cur, cur_size = [], 0
    if cur:
        chunks.append(cur)

    ref_shapes = {k: (v.shape, v.dtype) for k, v in phis[KEYS[0]].items()}
    logger.info(f"  청크 수: {len(chunks)} (청크당 {CHUNK_SIZE//1_000_000}M 파라미터)")

    # ── STEP 2: S = V^T V 청크 단위 누적 ─────────────────────
    logger.info(f"\nSTEP 2: Gram 행렬 S = V^T V 계산...")
    S = torch.zeros(n, n, dtype=torch.float32)
    log_interval = max(1, len(chunks) // 10)

    for chunk_idx, chunk_params in enumerate(chunks):
        chunk_param_keys = [pk for pk, _ in chunk_params]

        # 메모리 내에서 처리 (파일 I/O 없음!)
        cols = [
            torch.cat([phis[key][pk].float().reshape(-1) for pk in chunk_param_keys])
            for key in KEYS
        ]
        V_chunk = torch.stack(cols, dim=1)  # (chunk_size, n)
        del cols

        S += V_chunk.T @ V_chunk
        del V_chunk
        gc.collect()

        if (chunk_idx + 1) % log_interval == 0:
            logger.info(f"  [{chunk_idx+1}/{len(chunks)}] S 누적 중...")

    logger.info(f"  S 완료:\n{S}")

    # ── STEP 3: S^{-1/2} 계산 ─────────────────────────────────
    logger.info("\nSTEP 3: S^{-1/2} 계산...")
    eigenvalues, Q = torch.linalg.eigh(S)
    logger.info(f"  eigenvalues: {eigenvalues.tolist()}")
    eigenvalues_safe = torch.clamp(eigenvalues.abs(), min=EPS)
    S_inv_sqrt = Q @ torch.diag(eigenvalues_safe ** (-0.5)) @ Q.T
    del S, eigenvalues, eigenvalues_safe, Q
    gc.collect()
    logger.info(f"  S^{{-1/2}} 완료")

    # ── STEP 4: U = V S^{-1/2} 청크 단위 계산 → 임시 저장 ────
    logger.info(f"\nSTEP 4: U = V S^{{-1/2}} 계산 → 임시 저장...")

    for chunk_idx, chunk_params in enumerate(chunks):
        chunk_param_keys = [pk for pk, _ in chunk_params]

        cols = [
            torch.cat([phis[key][pk].float().reshape(-1) for pk in chunk_param_keys])
            for key in KEYS
        ]
        V_chunk = torch.stack(cols, dim=1).float()
        del cols

        U_chunk = V_chunk @ S_inv_sqrt  # (chunk_size, n)
        del V_chunk
        gc.collect()

        # 각 벡터의 청크를 임시 파일로 즉시 저장 → RAM 해제
        for vec_idx, key in enumerate(KEYS):
            tmp_path = os.path.join(tmp_dir, f"tmp_{key}_{chunk_idx:05d}.pt")
            torch.save(U_chunk[:, vec_idx].clone().bfloat16(), tmp_path)
        del U_chunk
        gc.collect()

        if (chunk_idx + 1) % log_interval == 0:
            logger.info(f"  [{chunk_idx+1}/{len(chunks)}] U 계산 중... {ram_usage()}")

    del S_inv_sqrt
    gc.collect()

    # ── STEP 5: phi 5개 해제 ──────────────────────────────────
    logger.info("\nSTEP 5: phi 메모리 해제...")
    for key in KEYS:
        del phis[key]
    del phis
    os_free()
    logger.info(f"  phi 해제 완료 | {ram_usage()}")

    # ── STEP 6: 임시 파일 합쳐서 최종 저장 ───────────────────
    logger.info("\nSTEP 6: 임시 청크 합쳐서 최종 저장...")
    saved_paths = {}

    for key in KEYS:
        # 청크 파일 순서대로 로드 → 이어붙이기
        chunk_tensors = []
        for chunk_idx in range(len(chunks)):
            tmp_path = os.path.join(tmp_dir, f"tmp_{key}_{chunk_idx:05d}.pt")
            chunk_tensors.append(
                torch.load(tmp_path, map_location="cpu", weights_only=True)
            )
            os.remove(tmp_path)  # 읽은 즉시 삭제

        full_flat = torch.cat(chunk_tensors)
        del chunk_tensors
        gc.collect()

        # param dict 복원
        result, offset = {}, 0
        for k in sorted(ref_shapes.keys()):
            shape, dtype = ref_shapes[k]
            numel = 1
            for s in shape:
                numel *= s
            result[k] = full_flat[offset:offset+numel].reshape(shape).to(dtype).clone()
            offset += numel

        # full_flat 해제 (result 완성 후)
        del full_flat
        os_free()

        # 저장 (phi_*.pt → merger.py 호환)
        save_path = os.path.join(out_dir, f"phi_{key}.pt")
        torch.save(result, save_path)
        del result
        os_free()

        size_mb = os.path.getsize(save_path) / (1024 ** 2)
        logger.info(f"  [{key}] → {save_path} ({size_mb:.1f} MB) | {ram_usage()}")
        saved_paths[key] = save_path

    # 임시 폴더 정리
    try:
        os.rmdir(tmp_dir)
    except OSError:
        pass

    # ── STEP 7: 직교성 검증 ────────────────────────────────────
    logger.info("\nSTEP 7: 직교성 검증 (샘플 파라미터 20개)...")
    _verify(KEYS, out_dir)

    # 메타 저장
    elapsed = round(time.time() - start, 1)
    manifest = {
        "method":      "Löwdin (Symmetric Orthogonalization)",
        "phi_dir":     PHI_DIR,
        "keys":        KEYS,
        "chunk_size":  CHUNK_SIZE,
        "created_at":  time.strftime("%Y-%m-%d %H:%M:%S"),
        "elapsed_sec": elapsed,
        "files":       saved_paths,
    }
    with open(os.path.join(OUTPUT_DIR, "manifest_lowdin.json"), "w") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    logger.info(f"\n{'='*60}")
    logger.info(f"  Löwdin 완료! 소요: {elapsed}초")
    logger.info(f"  저장: {len(saved_paths)}개 → {out_dir}/")
    logger.info(f"{'='*60}")
    logger.info("\n[ merger 연결 방법 ]")
    logger.info(f"  phi_dir = '{out_dir}'")

    return manifest


# ==============================================================================
# 직교성 검증
# ==============================================================================

def _verify(keys: list, out_dir: str):
    """샘플 파라미터로 코사인 유사도 확인 (≈ 0 이면 직교 ✅)"""
    first = torch.load(
        os.path.join(out_dir, f"phi_{keys[0]}.pt"),
        map_location="cpu", weights_only=True
    )
    sample_keys = sorted(first.keys())[:20]
    del first
    os_free()

    vecs = {}
    for key in keys:
        phi = torch.load(
            os.path.join(out_dir, f"phi_{key}.pt"),
            map_location="cpu", weights_only=True
        )
        vecs[key] = torch.cat([phi[k].float().reshape(-1) for k in sample_keys])
        del phi
        os_free()

    cos_list = []
    for i, k1 in enumerate(keys):
        for k2 in keys[i+1:]:
            n1 = vecs[k1].norm().item()
            n2 = vecs[k2].norm().item()
            if n1 < 1e-12 or n2 < 1e-12:
                continue
            cos = torch.dot(vecs[k1], vecs[k2]).item() / (n1 * n2)
            ok = "✅" if abs(cos) < 0.05 else "⚠️ "
            logger.info(f"  cos({k1}, {k2}) = {cos:.8f}  {ok}")
            cos_list.append(abs(cos))

    if cos_list:
        logger.info(f"  평균 |cos|: {sum(cos_list)/len(cos_list):.8f}")

    for key in keys:
        del vecs[key]
    os_free()


if __name__ == "__main__":
    run()

06:00:33 [INFO] ============================================================
06:00:33 [INFO]   Löwdin (Symmetric Orthogonalization)
06:00:33 [INFO]   phi 폴더  : /workspace/auto_personality/phi_vectors_llama
06:00:33 [INFO]   출력 폴더 : ./ortho_phi_vectors/lowdin
06:00:33 [INFO]   대상      : CON_High | NEU_High | EXT_High | AGR_High | OPN_High
06:00:33 [INFO] ============================================================
06:00:33 [INFO] 
STEP 1: phi 5개 로드 중... (총 ~75GB 예상)
06:00:33 [INFO]   [CON_High] 로드 중...
06:00:44 [INFO]   [CON_High] 완료 | RSS 15.5GB | 여유 1588.3GB
06:00:44 [INFO]   [NEU_High] 로드 중...
06:00:53 [INFO]   [NEU_High] 완료 | RSS 30.4GB | 여유 1575.2GB
06:00:53 [INFO]   [EXT_High] 로드 중...
06:01:03 [INFO]   [EXT_High] 완료 | RSS 45.4GB | 여유 1560.2GB
06:01:03 [INFO]   [AGR_High] 로드 중...
06:01:12 [INFO]   [AGR_High] 완료 | RSS 60.4GB | 여유 1544.6GB
06:01:12 [INFO]   [OPN_High] 로드 중...
06:01:21 [INFO]   [OPN_High] 완료 | RSS 75.3GB | 여유 1529.5GB
06:01:21 [INFO] 
  총 파라미터: 8,030,261,248
06:01:21 

KeyboardInterrupt: 

In [ ]:


import ctypes
import gc
import json
import logging
import os
import time

import torch

# ──────────────────────────────────────────────────────────────────────────────
torch.set_num_threads(8)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    libc = ctypes.CDLL("libc.so.6")
except Exception:
    libc = None

# ──────────────────────────────────────────────────────────────────────────────
# ★★★ 여기만 수정하세요 ★★★
# ──────────────────────────────────────────────────────────────────────────────

PHI_DIR    = ""
OUTPUT_DIR = "./ortho_phi_vectors"
KEYS       = ["CON_High", "NEU_High", "EXT_High", "AGR_High", "OPN_High"]

ALPHA     = 0.1      # norm constraint
LR        = 1e-2     # gradient descent β
MAX_ITER  = 300
TOL       = 1e-7
LOG_EVERY = 10

# 청크: 2B × float32 × 5텐서 = 40GB VRAM
# phi 5개 GPU 상주(75GB) + 여유 19.6GB
# 청크 연산: pi_c+pj_c+dc+vi_c+vj_c = 5텐서 × float32
# 안전 청크: 19GB / 5 / 4bytes = 950M → 500M으로 여유있게
CHUNK = 500_000_000  # 500M

# ──────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


# ==============================================================================
# 유틸
# ==============================================================================

def os_free():
    gc.collect()
    if libc:
        libc.malloc_trim(0)


def gpu_free():
    torch.cuda.empty_cache()


def phi_path(key: str) -> str:
    return os.path.join(PHI_DIR, f"phi_{key}.pt")


def ram_usage() -> str:
    try:
        import psutil
        proc = psutil.Process(os.getpid())
        rss  = proc.memory_info().rss / (1024 ** 3)
        vm   = psutil.virtual_memory()
        free_vram = torch.cuda.mem_get_info()[0] / 1024**3 if torch.cuda.is_available() else 0
        return f"RSS {rss:.1f}GB | RAM여유 {vm.available/1024**3:.1f}GB | VRAM여유 {free_vram:.1f}GB"
    except ImportError:
        return ""


def dict_to_flat_bf16(param_dict: dict, sorted_keys: list) -> torch.Tensor:
    """param dict → 1D bfloat16 CPU tensor"""
    return torch.cat([
        param_dict[k].bfloat16().reshape(-1)
        for k in sorted_keys
    ])


def to_gpu(phi_flats_cpu: list) -> list:
    """phi 5개를 GPU로 올림 (bfloat16 유지)"""
    logger.info(f"  phi → GPU 전송 중...")
    phi_flats_gpu = []
    for i, flat in enumerate(phi_flats_cpu):
        phi_flats_gpu.append(flat.to(DEVICE))
        free = torch.cuda.mem_get_info()[0] / 1024**3
        logger.info(f"  [{KEYS[i]}] GPU 전송 완료 | VRAM여유 {free:.1f}GB")
    return phi_flats_gpu


# ==============================================================================
# AWD 손실함수 + 수동 gradient
# ==============================================================================

def compute_awd_loss(
    delta: torch.Tensor,   # CPU bfloat16
    phi_flats: list,       # CPU bfloat16 list
    alpha: float,
) -> tuple:
    """
    논문 수식 11, 13, 14:
      L_O = (1/K(K-1)) Σ_{i≠j} |cos(τ_i-δ, τ_j-δ)|
      L_R = ‖δ‖₂
      L   = L_O + α * L_R

    gradient 수동 계산 (backward 없음):
      ∂|cos(vi,vj)|/∂δ = sign × [
        -(vj+vi)/(ni×nj) + cos×(vi/ni² + vj/nj²)
      ] / K(K-1)
      ∂L_R/∂δ = α × δ / ‖δ‖

    GPU 청크 전략:
      청크(2B)씩 GPU로 올려서 계산 → .item()/.cpu() 후 즉시 해제
      VRAM: 청크당 최대 40GB
    """
    K       = len(phi_flats)
    eps     = 1e-6
    n_pairs = K * (K - 1) // 2

    # ── PASS 1: 스칼라값 계산 (no_grad) ───────────────────────
    # dot, ni, nj를 스칼라로 누적 → VRAM에 tensor 남기지 않음
    dots    = []   # float 스칼라
    ni_vals = []   # float 스칼라
    nj_vals = []   # float 스칼라

    with torch.no_grad():
        for i in range(K):
            for j in range(i + 1, K):
                dot_val = 0.0
                ni_sq   = 0.0
                nj_sq   = 0.0

                for c in range(0, phi_flats[i].numel(), CHUNK):
                    # phi는 GPU 상주 → 슬라이스만
                    # delta만 CPU→GPU 전송
                    pi_c = phi_flats[i][c:c+CHUNK].float()  # GPU (이미 올라있음)
                    pj_c = phi_flats[j][c:c+CHUNK].float()  # GPU
                    dc   = delta[c:c+CHUNK].float().to(DEVICE)  # CPU→GPU

                    vi_c = pi_c - dc
                    vj_c = pj_c - dc

                    dot_val += (vi_c * vj_c).sum().item()
                    ni_sq   += vi_c.pow(2).sum().item()
                    nj_sq   += vj_c.pow(2).sum().item()

                    del pi_c, pj_c, dc, vi_c, vj_c
                    gpu_free()

                dots.append(dot_val)
                ni_vals.append(ni_sq ** 0.5)
                nj_vals.append(nj_sq ** 0.5)

        # delta norm (delta만 GPU로)
        norm_delta_sq = 0.0
        for c in range(0, delta.numel(), CHUNK):
            dc = delta[c:c+CHUNK].float().to(DEVICE)
            norm_delta_sq += dc.pow(2).sum().item()
            del dc
            gpu_free()
        norm_delta = norm_delta_sq ** 0.5

    # ── L_O, L 스칼라 계산 ────────────────────────────────────
    cos_sum  = 0.0
    pair_info = []

    for idx in range(n_pairs):
        ni  = ni_vals[idx]
        nj  = nj_vals[idx]
        dot = dots[idx]
        if ni < eps or nj < eps:
            pair_info.append(None)
            continue
        cos = dot / (ni * nj + eps)
        cos_sum += abs(cos)
        pair_info.append((cos, ni, nj, dot))

    # i,j 인덱스 복원
    ij_list = [(i, j) for i in range(K) for j in range(i+1, K)]

    L_O = cos_sum / max(n_pairs * 2, 1)
    L_R = norm_delta
    L   = L_O + alpha * L_R

    # ── PASS 2: gradient 수동 계산 ────────────────────────────
    # grad_accum: CPU bfloat16 (delta와 동일 dtype/shape)
    grad_accum = torch.zeros(delta.numel(), dtype=torch.bfloat16)

    # ∂L_O/∂δ
    for idx, info in enumerate(pair_info):
        if info is None:
            continue
        cos, ni, nj, dot = info
        i, j = ij_list[idx]
        sign = 1.0 if dot >= 0 else -1.0

        for c in range(0, phi_flats[i].numel(), CHUNK):
            pi_c = phi_flats[i][c:c+CHUNK].float()       # GPU 상주
            pj_c = phi_flats[j][c:c+CHUNK].float()       # GPU 상주
            dc   = delta[c:c+CHUNK].float().to(DEVICE)   # CPU→GPU

            vi_c = pi_c - dc
            vj_c = pj_c - dc

            # ∂|cos|/∂δ = sign × ∂cos/∂δ
            # ∂cos/∂δ = -(vj+vi)/(ni×nj) + cos×(vi/ni² + vj/nj²)
            grad_c = -(
                sign * (vi_c + vj_c) / (ni * nj + eps)
                - sign * cos * vi_c / (ni ** 2 + eps)
                - sign * cos * vj_c / (nj ** 2 + eps)
            ) / max(n_pairs * 2, 1)

            # CPU로 이동 후 즉시 해제
            grad_accum[c:c+CHUNK] += grad_c.cpu().bfloat16()

            del pi_c, pj_c, dc, vi_c, vj_c, grad_c
            gpu_free()

    # ∂L_R/∂δ = α × δ / ‖δ‖ (delta만 GPU로)
    if norm_delta > eps:
        for c in range(0, delta.numel(), CHUNK):
            dc = delta[c:c+CHUNK].float().to(DEVICE)
            grad_accum[c:c+CHUNK] += (
                alpha * dc / (norm_delta + eps)
            ).cpu().bfloat16()
            del dc
            gpu_free()

    return L, L_O, L_R, grad_accum


# ==============================================================================
# δ 최적화 (논문 Algorithm 1)
# ==============================================================================

def optimize_delta(
    phi_flats: list,
    total_numel: int,
    alpha: float,
    lr: float,
    max_iter: int,
    tol: float,
) -> torch.Tensor:
    """
    δ⁰ ← 0
    δⁿ = δⁿ⁻¹ - β × ∇L  (논문 Algorithm 1)
    """
    # delta: CPU bfloat16
    delta = torch.zeros(total_numel, dtype=torch.bfloat16)

    prev_loss = float("inf")
    logger.info(f"  δ: CPU bfloat16 {total_numel:,} | {ram_usage()}")
    logger.info(f"  Algorithm 1: δⁿ = δⁿ⁻¹ - β∇L | β={lr} | α={alpha}")
    logger.info(f"  CHUNK={CHUNK:,} | DEVICE={DEVICE}")

    for it in range(max_iter):
        t0 = time.time()

        L, L_O, L_R, grad = compute_awd_loss(delta, phi_flats, alpha)

        # δⁿ = δⁿ⁻¹ - β × ∇L (in-place → 추가 메모리 없음)
        delta -= lr * grad
        del grad
        os_free()

        loss_val = float(L)

        if (it + 1) % LOG_EVERY == 0:
            elapsed = time.time() - t0
            logger.info(
                f"  iter [{it+1:4d}/{max_iter}] "
                f"loss={loss_val:.8f} L_O={L_O:.8f} "
                f"norm_δ={L_R:.6f} | {elapsed:.1f}s | {ram_usage()}"
            )

        if abs(prev_loss - loss_val) < tol and it > 10:
            logger.info(f"  ✅ 수렴! iter={it+1} | loss={loss_val:.10f}")
            break

        prev_loss = loss_val

    logger.info(f"  최종 norm(δ) = {float(L_R):.6f}")
    return delta


# ==============================================================================
# 코사인 유사도 측정 (검증용)
# ==============================================================================

def measure_cosine(phi_flats: list, delta: torch.Tensor = None) -> float:
    """GPU 청크로 |cos| 측정"""
    K   = len(phi_flats)
    eps = 1e-8
    cos_list = []

    with torch.no_grad():
        for i in range(K):
            for j in range(i + 1, K):
                dot_val = 0.0
                ni_sq   = 0.0
                nj_sq   = 0.0

                for c in range(0, phi_flats[i].numel(), CHUNK):
                    pi_c = phi_flats[i][c:c+CHUNK].float()  # GPU 상주
                    pj_c = phi_flats[j][c:c+CHUNK].float()  # GPU 상주

                    if delta is not None:
                        dc   = delta[c:c+CHUNK].float().to(DEVICE)  # CPU→GPU
                        pi_c = pi_c - dc
                        pj_c = pj_c - dc
                        del dc

                    dot_val += (pi_c * pj_c).sum().item()
                    ni_sq   += pi_c.pow(2).sum().item()
                    nj_sq   += pj_c.pow(2).sum().item()

                    del pi_c, pj_c
                    gpu_free()

                ni  = ni_sq ** 0.5
                nj  = nj_sq ** 0.5
                if ni < eps or nj < eps:
                    continue

                cos = abs(dot_val) / (ni * nj)
                cos_list.append(cos)
                logger.info(
                    f"  |cos({KEYS[i]}, {KEYS[j]})| = {cos:.6f}  "
                    f"{'✅' if cos < 0.1 else '⚠️ '}"
                )

    return sum(cos_list) / len(cos_list) if cos_list else 0.0


# ==============================================================================
# 메인
# ==============================================================================

def run():
    out_dir = os.path.join(OUTPUT_DIR, "awd")
    os.makedirs(out_dir, exist_ok=True)

    logger.info("=" * 60)
    logger.info("  AWD (Adaptive Weight Disentanglement)")
    logger.info("  Xiong et al., arXiv:2411.18729, 2024")
    logger.info(f"  phi 폴더  : {PHI_DIR}")
    logger.info(f"  출력 폴더 : {out_dir}")
    logger.info(f"  대상      : {' | '.join(KEYS)}")
    logger.info(f"  α={ALPHA} | lr={LR} | max_iter={MAX_ITER}")
    logger.info(f"  DEVICE={DEVICE} | CHUNK={CHUNK:,}")
    logger.info("=" * 60)

    missing = [k for k in KEYS if not os.path.exists(phi_path(k))]
    if missing:
        logger.error(f"phi 파일 없음: {missing}")
        return None

    start = time.time()

    # ── STEP 1: phi 5개 로드 (CPU) ────────────────────────────
    logger.info(f"\nSTEP 1: phi {len(KEYS)}개 로드 (CPU)...")
    phi_flats   = []
    sorted_keys = None
    ref_shapes  = None

    for key in KEYS:
        logger.info(f"  [{key}] 로드 중...")
        phi_dict = torch.load(phi_path(key), map_location="cpu", weights_only=True)
        if sorted_keys is None:
            sorted_keys = sorted(phi_dict.keys())
            ref_shapes  = {k: (v.shape, v.dtype) for k, v in phi_dict.items()}
        flat = dict_to_flat_bf16(phi_dict, sorted_keys)
        phi_flats.append(flat)
        del phi_dict
        os_free()
        logger.info(f"  [{key}] 완료 | {ram_usage()}")

    total_numel = phi_flats[0].numel()
    logger.info(f"  파라미터 수: {total_numel:,} | {ram_usage()}")

    # ── STEP 1.5: phi GPU 전송 ───────────────────────────────
    logger.info("\nSTEP 1.5: phi 5개 GPU로 전송...")
    phi_flats_gpu = to_gpu(phi_flats)
    del phi_flats
    os_free()
    logger.info(f"  GPU 전송 완료 | {ram_usage()}")

    # ── STEP 2: 직교화 전 측정 ────────────────────────────────
    logger.info("\nSTEP 2: 직교화 전 코사인 유사도...")
    avg_before = measure_cosine(phi_flats_gpu, delta=None)
    logger.info(f"  평균 |cos|: {avg_before:.6f}")

    # ── STEP 3: δ 최적화 ──────────────────────────────────────
    logger.info(f"\nSTEP 3: δ 최적화 (Algorithm 1)...")
    delta = optimize_delta(
        phi_flats_gpu, total_numel, ALPHA, LR, MAX_ITER, TOL
    )

    # ── STEP 4: 직교화 후 측정 ────────────────────────────────
    logger.info("\nSTEP 4: 직교화 후 코사인 유사도...")
    avg_after = measure_cosine(phi_flats_gpu, delta=delta)
    reduction = (avg_before - avg_after) / avg_before * 100
    logger.info(f"  평균 |cos|: {avg_after:.6f}")
    logger.info(f"  간섭 감소: {reduction:.1f}%  {'✅' if reduction > 0 else '❌'}")

    # ── STEP 5: 저장 ──────────────────────────────────────────
    logger.info("\nSTEP 5: phi_i - δ 저장...")
    saved_paths = {}

    for i, key in enumerate(KEYS):
        ortho_flat = phi_flats_gpu[i].cpu().float() - delta.float()

        result, offset = {}, 0
        for k in sorted_keys:
            shape, dtype = ref_shapes[k]
            numel = 1
            for s in shape:
                numel *= s
            result[k] = ortho_flat[offset:offset+numel].reshape(shape).to(dtype).clone()
            offset += numel

        del ortho_flat
        os_free()

        save_path = os.path.join(out_dir, f"phi_{key}.pt")
        torch.save(result, save_path)
        del result
        os_free()

        size_mb = os.path.getsize(save_path) / (1024 ** 2)
        logger.info(f"  [{key}] → {save_path} ({size_mb:.1f} MB) | {ram_usage()}")
        saved_paths[key] = save_path

    del phi_flats_gpu, delta
    os_free()

    elapsed = round(time.time() - start, 1)
    manifest = {
        "method":      "AWD (Xiong et al., arXiv:2411.18729, 2024)",
        "phi_dir":     PHI_DIR,
        "keys":        KEYS,
        "alpha":       ALPHA,
        "lr":          LR,
        "max_iter":    MAX_ITER,
        "cos_before":  round(avg_before, 6),
        "cos_after":   round(avg_after, 6),
        "interference_reduction_pct": round(reduction, 2),
        "created_at":  time.strftime("%Y-%m-%d %H:%M:%S"),
        "elapsed_sec": elapsed,
        "files":       saved_paths,
    }
    with open(os.path.join(OUTPUT_DIR, "manifest_awd.json"), "w") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    logger.info(f"\n{'='*60}")
    logger.info(f"  AWD 완료! 소요: {elapsed}초")
    logger.info(f"  간섭 감소: {reduction:.1f}%")
    logger.info(f"{'='*60}")
    return manifest


if __name__ == "__main__":
    run()

11:56:34 [INFO] ============================================================
11:56:34 [INFO]   AWD (Adaptive Weight Disentanglement)
11:56:34 [INFO]   Xiong et al., arXiv:2411.18729, 2024
11:56:34 [INFO]   phi 폴더  : /workspace/auto_personality/phi_vectors_llama
11:56:34 [INFO]   출력 폴더 : ./ortho_phi_vectors/awd
11:56:34 [INFO]   대상      : CON_High | NEU_High | EXT_High | AGR_High | OPN_High
11:56:34 [INFO]   α=0.1 | lr=0.01 | max_iter=300
11:56:34 [INFO]   DEVICE=cuda | CHUNK=500,000,000
11:56:34 [INFO] ============================================================
11:56:34 [INFO] 
STEP 1: phi 5개 로드 (CPU)...
11:56:34 [INFO]   [CON_High] 로드 중...
11:56:46 [INFO]   [CON_High] 완료 | RSS 15.4GB | RAM여유 1632.0GB | VRAM여유 94.4GB
11:56:46 [INFO]   [NEU_High] 로드 중...
11:56:59 [INFO]   [NEU_High] 완료 | RSS 30.5GB | RAM여유 1615.9GB | VRAM여유 94.4GB
11:56:59 [INFO]   [EXT_High] 로드 중...
11:57:11 [INFO]   [EXT_High] 완료 | RSS 45.4GB | RAM여유 1600.9GB | VRAM여유 94.4GB
11:57:11 [INFO]   [AGR_High] 로드 중...
11:57

In [ ]:
#!/usr/bin/env python3
"""
cpu_oom_test.py
CPU RAM을 조금씩 채워서 OOM 발생 시점을 확인하는 테스트.
"""
import torch
import psutil
import gc

def ram_usage():
    vm = psutil.virtual_memory()
    proc = psutil.Process()
    rss = proc.memory_info().rss / 1024**3
    return f"RSS {rss:.1f}GB | 여유 {vm.available/1024**3:.1f}GB | 사용률 {vm.percent:.1f}%"

print("=" * 50)
print("CPU RAM OOM 테스트 시작")
print(f"초기 상태: {ram_usage()}")
print("=" * 50)

chunks = []
CHUNK_GB = 100  # 한 번에 5GB씩 할당
count = 0

try:
    while True:
        # 5GB짜리 float32 텐서 (CPU)
        numel = int(CHUNK_GB * 1024**3 / 4)  # float32 = 4bytes
        t = torch.zeros(numel, dtype=torch.float32)
        chunks.append(t)
        count += 1
        total_allocated = count * CHUNK_GB
        print(f"  [{count}] +{CHUNK_GB}GB 할당 (누적 {total_allocated}GB) | {ram_usage()}")

except MemoryError as e:
    print(f"\n💀 MemoryError 발생! (Python 레벨)")
    print(f"  총 할당: {count * CHUNK_GB}GB")
    print(f"  {ram_usage()}")

except RuntimeError as e:
    print(f"\n💀 RuntimeError 발생! (PyTorch 레벨)")
    print(f"  {e}")
    print(f"  총 할당: {count * CHUNK_GB}GB")
    print(f"  {ram_usage()}")

except Exception as e:
    print(f"\n💀 기타 에러: {type(e).__name__}: {e}")
    print(f"  총 할당: {count * CHUNK_GB}GB")
    print(f"  {ram_usage()}")

finally:
    print("\n해제 중...")
    del chunks
    gc.collect()
    print(f"해제 완료 | {ram_usage()}")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import ctypes
import gc
import json
import logging
import os
import time

import torch

# ──────────────────────────────────────────────────────────────────────────────
torch.set_num_threads(8)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    libc = ctypes.CDLL("libc.so.6")
except Exception:
    libc = None

# ──────────────────────────────────────────────────────────────────────────────
# ★★★ 여기만 수정하세요 ★★★
# ──────────────────────────────────────────────────────────────────────────────

PHI_DIR    = ""
OUTPUT_DIR = "./ortho_phi_vectors"
KEYS       = ["CON_High", "NEU_High", "EXT_High", "AGR_High", "OPN_High"]

ALPHA     = 0.1
LR        = 1e-5
MAX_ITER  = 300
TOL       = 1e-7
LOG_EVERY = 1   # iter마다 로그

# ──────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


# ==============================================================================
# 유틸
# ==============================================================================

def os_free():
    gc.collect()
    torch.cuda.empty_cache()
    if libc:
        libc.malloc_trim(0)


def phi_path(key: str) -> str:
    return os.path.join(PHI_DIR, f"phi_{key}.pt")


def mem_usage() -> str:
    try:
        import psutil
        rss  = psutil.Process(os.getpid()).memory_info().rss / 1024**3
        free_v, total_v = torch.cuda.mem_get_info()
        used_v = (total_v - free_v) / 1024**3
        return f"RAM {rss:.1f}GB | VRAM {used_v:.1f}/{total_v/1024**3:.0f}GB"
    except Exception:
        return ""


def dict_to_flat(param_dict: dict, sorted_keys: list) -> torch.Tensor:
    """param dict → 1D bfloat16 CPU tensor"""
    return torch.cat([
        param_dict[k].bfloat16().reshape(-1)
        for k in sorted_keys
    ])


# ==============================================================================
# AWD 손실 + 수동 gradient (논문 수식 그대로)
# ==============================================================================

def compute_awd_loss(delta: torch.Tensor, phi_flats: list, alpha: float) -> tuple:
    """
    논문 수식 11, 13, 14:
      L_O = (1/K(K-1)) Σ_{i≠j} |cos(τ_i-δ, τ_j-δ)|
      L_R = ‖δ‖₂
      L   = L_O + α * L_R

    수동 gradient (쌍별 GPU 올렸다가 즉시 해제):
      PASS 1: 스칼라값 계산
      PASS 2: gradient 누적
    """
    K       = len(phi_flats)
    eps     = 1e-6
    n_pairs = K * (K - 1) // 2
    ij_list = [(i, j) for i in range(K) for j in range(i+1, K)]

    # gradient 누적 (CPU bfloat16)
    grad_accum = torch.zeros_like(delta)

    # ── PASS 1: 스칼라값 계산 ─────────────────────────────────
    cos_sum   = 0.0
    pair_info = []

    with torch.no_grad():
        for idx, (i, j) in enumerate(ij_list):
            logger.info(f"    PASS1 [{idx+1}/{n_pairs}] ({KEYS[i]}, {KEYS[j]}) | {mem_usage()}")

            phi_i   = phi_flats[i].to(DEVICE)
            phi_j   = phi_flats[j].to(DEVICE)
            delta_g = delta.to(DEVICE)

            vi = phi_i - delta_g
            vj = phi_j - delta_g

            ni  = vi.norm().item()
            nj  = vj.norm().item()
            dot = (vi * vj).sum().item()

            del phi_i, phi_j, delta_g, vi, vj
            os_free()

            if ni < eps or nj < eps:
                pair_info.append(None)
                continue

            cos = dot / (ni * nj + eps)
            cos_sum += abs(cos)
            pair_info.append((cos, ni, nj, dot))

        norm_delta = delta.to(DEVICE).norm().item()
        os_free()

    # L_O, L_R, L 계산
    L_O = cos_sum / max(n_pairs * 2, 1)
    L_R = norm_delta
    L   = L_O + alpha * L_R

    # ── PASS 2: gradient 수동 계산 ────────────────────────────
    with torch.no_grad():
        for idx, info in enumerate(pair_info):
            if info is None:
                continue
            cos, ni, nj, dot = info
            i, j = ij_list[idx]
            sign = 1.0 if dot >= 0 else -1.0

            logger.info(f"    PASS2 [{idx+1}/{n_pairs}] ({KEYS[i]}, {KEYS[j]}) grad 계산 중...")

            phi_i   = phi_flats[i].to(DEVICE)
            phi_j   = phi_flats[j].to(DEVICE)
            delta_g = delta.to(DEVICE)

            vi = phi_i - delta_g
            vj = phi_j - delta_g

            # ∂|cos(vi,vj)|/∂δ = sign × ∂cos/∂δ
            # ∂cos/∂δ = -(vi+vj)/(ni×nj) + cos×(vi/ni² + vj/nj²)
            grad_ij = -(
                sign * (vi + vj) / (ni * nj + eps)
                - sign * cos * vi / (ni ** 2 + eps)
                - sign * cos * vj / (nj ** 2 + eps)
            ) / max(n_pairs * 2, 1)

            grad_accum += grad_ij.cpu()

            del phi_i, phi_j, delta_g, vi, vj, grad_ij
            os_free()

        # ∂L_R/∂δ = α × δ / ‖δ‖
        if norm_delta > eps:
            grad_accum += (alpha * delta / (norm_delta + eps))

    return L, L_O, L_R, grad_accum


# ==============================================================================
# δ 최적화 (논문 Algorithm 1)
# ==============================================================================

def optimize_delta(phi_flats: list, total_numel: int,
                   alpha: float, lr: float, max_iter: int, tol: float) -> torch.Tensor:
    """
    δ⁰ ← 0
    δⁿ = δⁿ⁻¹ - β × ∇L  (논문 Algorithm 1 그대로)
    """
    delta = torch.zeros(total_numel, dtype=torch.bfloat16)  # CPU

    prev_loss = float("inf")
    logger.info(f"  δ: CPU bfloat16 {total_numel:,} | {mem_usage()}")
    logger.info(f"  Algorithm 1: δⁿ = δⁿ⁻¹ - β∇L | β={lr} | α={alpha}")

    for it in range(max_iter):
        t0 = time.time()
        logger.info(f"\n  ── iter [{it+1:4d}/{max_iter}] ──────────────────")

        L, L_O, L_R, grad = compute_awd_loss(delta, phi_flats, alpha)
        loss_val = float(L)

        # δⁿ = δⁿ⁻¹ - β × ∇L
        delta -= lr * grad
        del grad
        os_free()

        elapsed = time.time() - t0
        logger.info(
            f"  iter [{it+1:4d}/{max_iter}] ✅ "
            f"loss={loss_val:.6f} L_O={L_O:.6f} norm_δ={L_R:.4f} | "
            f"{elapsed:.1f}s | {mem_usage()}"
        )

        if abs(prev_loss - loss_val) < tol and it > 10:
            logger.info(f"  ✅ 수렴! iter={it+1}")
            break

        prev_loss = loss_val

    return delta


# ==============================================================================
# 코사인 유사도 측정
# ==============================================================================

def measure_cosine(phi_flats: list, delta: torch.Tensor = None) -> float:
    K, eps, cos_list = len(phi_flats), 1e-8, []
    ij_list = [(i, j) for i in range(K) for j in range(i+1, K)]

    with torch.no_grad():
        for i, j in ij_list:
            phi_i   = phi_flats[i].to(DEVICE)
            phi_j   = phi_flats[j].to(DEVICE)
            delta_g = delta.to(DEVICE) if delta is not None else None

            vi = phi_i - delta_g if delta_g is not None else phi_i
            vj = phi_j - delta_g if delta_g is not None else phi_j

            ni  = vi.norm().item()
            nj  = vj.norm().item()
            dot = (vi * vj).sum().item()

            del phi_i, phi_j, vi, vj
            if delta_g is not None:
                del delta_g
            os_free()

            if ni < eps or nj < eps:
                continue

            cos = abs(dot) / (ni * nj)
            cos_list.append(cos)
            logger.info(
                f"  |cos({KEYS[i]}, {KEYS[j]})| = {cos:.6f}  "
                f"{'✅' if cos < 0.1 else '⚠️ '}"
            )

    return sum(cos_list) / len(cos_list) if cos_list else 0.0


# ==============================================================================
# 메인
# ==============================================================================

def run():
    out_dir = os.path.join(OUTPUT_DIR, "awd")
    os.makedirs(out_dir, exist_ok=True)

    logger.info("=" * 60)
    logger.info("  AWD (Adaptive Weight Disentanglement)")
    logger.info("  Xiong et al., arXiv:2411.18729, 2024")
    logger.info(f"  phi 폴더  : {PHI_DIR}")
    logger.info(f"  출력 폴더 : {out_dir}")
    logger.info(f"  대상      : {' | '.join(KEYS)}")
    logger.info(f"  α={ALPHA} | lr={LR} | max_iter={MAX_ITER}")
    logger.info(f"  DEVICE={DEVICE} | phi+delta CPU, 쌍별 GPU 처리")
    logger.info("=" * 60)

    missing = [k for k in KEYS if not os.path.exists(phi_path(k))]
    if missing:
        logger.error(f"phi 파일 없음: {missing}")
        return None

    start = time.time()

    # ── STEP 1: phi 로드 (CPU) ────────────────────────────────
    logger.info(f"\nSTEP 1: phi {len(KEYS)}개 로드 (CPU bfloat16)...")
    phi_flats   = []
    sorted_keys = None
    ref_shapes  = None

    for key in KEYS:
        logger.info(f"  [{key}] 로드 중...")
        phi_dict = torch.load(phi_path(key), map_location="cpu", weights_only=True)
        if sorted_keys is None:
            sorted_keys = sorted(phi_dict.keys())
            ref_shapes  = {k: (v.shape, v.dtype) for k, v in phi_dict.items()}
        phi_flats.append(dict_to_flat(phi_dict, sorted_keys))
        del phi_dict
        gc.collect()
        logger.info(f"  [{key}] 완료 | {mem_usage()}")

    total_numel = phi_flats[0].numel()
    logger.info(f"  파라미터 수: {total_numel:,}")

    # ── STEP 2: 직교화 전 측정 ────────────────────────────────
    logger.info("\nSTEP 2: 직교화 전 코사인 유사도...")
    avg_before = measure_cosine(phi_flats, delta=None)
    logger.info(f"  평균 |cos|: {avg_before:.6f}")

    # ── STEP 3: δ 최적화 ──────────────────────────────────────
    logger.info(f"\nSTEP 3: δ 최적화 (Algorithm 1)...")
    delta = optimize_delta(phi_flats, total_numel, ALPHA, LR, MAX_ITER, TOL)

    # ── STEP 4: 직교화 후 측정 ────────────────────────────────
    logger.info("\nSTEP 4: 직교화 후 코사인 유사도...")
    avg_after = measure_cosine(phi_flats, delta=delta)
    reduction = (avg_before - avg_after) / avg_before * 100
    logger.info(f"  평균 |cos|: {avg_after:.6f}")
    logger.info(f"  간섭 감소: {reduction:.1f}%  {'✅' if reduction > 0 else '❌'}")

    # ── STEP 5: 저장 ──────────────────────────────────────────
    logger.info("\nSTEP 5: phi_i - δ 저장...")
    saved_paths = {}

    for i, key in enumerate(KEYS):
        ortho_flat = phi_flats[i].float() - delta.float()

        result, offset = {}, 0
        for k in sorted_keys:
            shape, dtype = ref_shapes[k]
            numel = 1
            for s in shape:
                numel *= s
            result[k] = ortho_flat[offset:offset+numel].reshape(shape).to(dtype).clone()
            offset += numel

        del ortho_flat
        gc.collect()

        save_path = os.path.join(out_dir, f"phi_{key}.pt")
        torch.save(result, save_path)
        del result
        gc.collect()

        size_mb = os.path.getsize(save_path) / 1024**2
        logger.info(f"  [{key}] → {save_path} ({size_mb:.1f} MB)")
        saved_paths[key] = save_path

    del phi_flats, delta
    os_free()

    elapsed = round(time.time() - start, 1)
    manifest = {
        "method":      "AWD (Xiong et al., arXiv:2411.18729, 2024)",
        "phi_dir":     PHI_DIR,
        "keys":        KEYS,
        "alpha":       ALPHA,
        "lr":          LR,
        "max_iter":    MAX_ITER,
        "cos_before":  round(avg_before, 6),
        "cos_after":   round(avg_after, 6),
        "interference_reduction_pct": round(reduction, 2),
        "created_at":  time.strftime("%Y-%m-%d %H:%M:%S"),
        "elapsed_sec": elapsed,
        "files":       saved_paths,
    }
    with open(os.path.join(OUTPUT_DIR, "manifest_awd.json"), "w") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    logger.info(f"\n{'='*60}")
    logger.info(f"  AWD 완료! 소요: {elapsed}초")
    logger.info(f"  간섭 감소: {reduction:.1f}%")
    logger.info(f"{'='*60}")
    return manifest


if __name__ == "__main__":
    run()

07:21:45 [INFO] ============================================================
07:21:45 [INFO]   AWD (Adaptive Weight Disentanglement)
07:21:45 [INFO]   Xiong et al., arXiv:2411.18729, 2024
07:21:45 [INFO]   phi 폴더  : /workspace/auto_personality/phi_vectors_llama
07:21:45 [INFO]   출력 폴더 : ./ortho_phi_vectors/awd
07:21:45 [INFO]   대상      : CON_High | NEU_High | EXT_High | AGR_High | OPN_High
07:21:45 [INFO]   α=0.1 | lr=0.01 | max_iter=300
07:21:45 [INFO]   DEVICE=cuda | phi+delta CPU, 쌍별 GPU 처리
07:21:45 [INFO] ============================================================
07:21:45 [INFO] 
STEP 1: phi 5개 로드 (CPU bfloat16)...
07:21:45 [INFO]   [CON_High] 로드 중...
07:22:16 [INFO]   [CON_High] 완료 | RAM 15.8GB | VRAM 0.5/140GB
07:22:16 [INFO]   [NEU_High] 로드 중...
07:22:44 [INFO]   [NEU_High] 완료 | RAM 30.9GB | VRAM 0.5/140GB
07:22:44 [INFO]   [EXT_High] 로드 중...
07:23:16 [INFO]   [EXT_High] 완료 | RAM 45.8GB | VRAM 0.5/140GB
07:23:16 [INFO]   [AGR_High] 로드 중...
07:23:45 [INFO]   [AGR_High] 완료 | RAM

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
run_awd_resume.py  —  AWD 21iter 결과에서 이어서 최적화

δ 복원: phi_i(원본) - phi_i_ortho(저장된 것) = δ
이어서: δ_21 → δ_300 추가 최적화

[ Jupyter 사용법 ]
  import run_awd_resume as r
  r.PHI_DIR     = "/workspace/auto_personality/phi_vectors_llama"
  r.ORTHO_DIR   = "./ortho_phi_vectors/awd"   # 21iter 저장 폴더
  r.OUTPUT_DIR  = "./ortho_phi_vectors"
  r.DONE_ITER   = 21                           # 이전에 완료한 iter
  r.ADD_ITER    = 279                          # 추가할 iter (21+279=300)
  r.run()
"""

import ctypes, gc, json, logging, os, time
import torch

torch.set_num_threads(8)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    libc = ctypes.CDLL("libc.so.6")
except Exception:
    libc = None

# ──────────────────────────────────────────────────────────────────────────────
# ★★★ 여기만 수정하세요 ★★★
# ──────────────────────────────────────────────────────────────────────────────
PHI_DIR    = ""
ORTHO_DIR  = "./ortho_phi_vectors/awd"   # 21iter 저장된 phi_i - δ
OUTPUT_DIR = "./ortho_phi_vectors"
KEYS       = ["CON_High", "NEU_High", "EXT_High", "AGR_High", "OPN_High"]

DONE_ITER  = 21     # 이전에 완료한 iter
ADD_ITER   = 279    # 추가할 iter → 총 300iter
ALPHA      = 0.1
LR         = 0.01
TOL        = 1e-7   # 이번엔 사실상 ADD_ITER까지 돌도록
# ──────────────────────────────────────────────────────────────────────────────

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s",
                    datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)


def os_free():
    gc.collect()
    torch.cuda.empty_cache()
    if libc: libc.malloc_trim(0)

def mem_usage():
    try:
        import psutil
        rss = psutil.Process(os.getpid()).memory_info().rss / 1024**3
        free_v, total_v = torch.cuda.mem_get_info()
        return f"RAM {rss:.1f}GB | VRAM {(total_v-free_v)/1024**3:.1f}/{total_v/1024**3:.0f}GB"
    except: return ""

def dict_to_flat(param_dict, sorted_keys):
    return torch.cat([param_dict[k].bfloat16().reshape(-1) for k in sorted_keys])

def phi_path(key): return os.path.join(PHI_DIR, f"phi_{key}.pt")
def ortho_path(key): return os.path.join(ORTHO_DIR, f"phi_{key}.pt")


# ==============================================================================
# δ 복원
# ==============================================================================
def recover_delta(sorted_keys):
    """δ = phi_i(원본) - phi_i(직교화본)"""
    key = KEYS[0]
    logger.info(f"  δ 복원 중 ({key} 사용)...")

    orig  = torch.load(phi_path(key),  map_location="cpu", weights_only=True)
    ortho = torch.load(ortho_path(key), map_location="cpu", weights_only=True)

    flat_orig  = dict_to_flat(orig,  sorted_keys).float()
    flat_ortho = dict_to_flat(ortho, sorted_keys).float()
    del orig, ortho; os_free()

    delta = (flat_orig - flat_ortho)   # float32
    del flat_orig, flat_ortho; os_free()

    logger.info(f"  δ 복원 완료 | norm_δ={delta.norm().item():.6f}")
    return delta


# ==============================================================================
# AWD loss + gradient (원본 코드와 동일)
# ==============================================================================
def compute_awd_loss(delta, phi_flats, alpha):
    K    = len(phi_flats)
    eps  = 1e-6
    KK1  = K * (K - 1)
    ij_list = [(i,j) for i in range(K) for j in range(i+1, K)]
    grad_accum = torch.zeros(delta.numel(), dtype=torch.float32)

    # PASS 1
    cos_sum, pair_info = 0.0, []
    with torch.no_grad():
        for idx, (i,j) in enumerate(ij_list):
            logger.info(f"    PASS1 [{idx+1}/{len(ij_list)}] ({KEYS[i]}, {KEYS[j]}) | {mem_usage()}")
            phi_i   = phi_flats[i].to(DEVICE)
            phi_j   = phi_flats[j].to(DEVICE)
            delta_g = delta.to(DEVICE).bfloat16()
            vi = phi_i - delta_g
            vj = phi_j - delta_g
            ni = vi.norm().item(); nj = vj.norm().item()
            dot = (vi * vj).sum().item()
            del phi_i, phi_j, delta_g, vi, vj; os_free()
            if ni < eps or nj < eps: pair_info.append(None); continue
            cos = dot / (ni * nj + eps)
            cos_sum += 2 * abs(cos)
            pair_info.append((cos, ni, nj, dot))
        norm_delta = delta.to(DEVICE).norm().item(); os_free()

    L_O = cos_sum / KK1
    L_R = norm_delta
    L   = L_O + alpha * L_R

    # PASS 2
    with torch.no_grad():
        for idx, info in enumerate(pair_info):
            if info is None: continue
            cos, ni, nj, dot = info
            i, j = ij_list[idx]
            sign  = 1.0 if dot >= 0 else -1.0
            scale = 2.0 / KK1
            logger.info(f"    PASS2 [{idx+1}/{len(ij_list)}] ({KEYS[i]}, {KEYS[j]}) grad 계산 중...")
            phi_i   = phi_flats[i].to(DEVICE)
            phi_j   = phi_flats[j].to(DEVICE)
            delta_g = delta.to(DEVICE).bfloat16()

            # 항1
            tmp = (phi_i + phi_j - 2 * delta_g) * (sign / (ni * nj + eps))
            grad_accum -= scale * tmp.cpu().float(); del tmp; torch.cuda.empty_cache()
            # 항2
            tmp = (phi_i - delta_g) * (sign * cos / (ni ** 2 + eps))
            grad_accum += scale * tmp.cpu().float(); del tmp; torch.cuda.empty_cache()
            # 항3
            tmp = (phi_j - delta_g) * (sign * cos / (nj ** 2 + eps))
            grad_accum += scale * tmp.cpu().float(); del tmp; torch.cuda.empty_cache()

            del phi_i, phi_j, delta_g; os_free()

        if norm_delta > eps:
            grad_accum += alpha * delta / (norm_delta + eps)

    return L, L_O, L_R, grad_accum


# ==============================================================================
# 코사인 측정
# ==============================================================================
def measure_cosine(phi_flats, delta=None):
    K, eps, cos_list = len(phi_flats), 1e-8, []
    for i,j in [(i,j) for i in range(K) for j in range(i+1,K)]:
        phi_i   = phi_flats[i].to(DEVICE)
        phi_j   = phi_flats[j].to(DEVICE)
        delta_g = delta.to(DEVICE).bfloat16() if delta is not None else None
        vi = phi_i - delta_g if delta_g is not None else phi_i
        vj = phi_j - delta_g if delta_g is not None else phi_j
        ni = vi.norm().item(); nj = vj.norm().item()
        dot = (vi * vj).sum().item()
        del phi_i, phi_j, vi, vj
        if delta_g is not None: del delta_g
        os_free()
        if ni < eps or nj < eps: continue
        cos = abs(dot) / (ni * nj)
        cos_list.append(cos)
        logger.info(f"  |cos({KEYS[i]}, {KEYS[j]})| = {cos:.6f}  {'✅' if cos<0.1 else '⚠️ '}")
    return sum(cos_list)/len(cos_list) if cos_list else 0.0


# ==============================================================================
# 메인
# ==============================================================================
def run():
    total_iter = DONE_ITER + ADD_ITER
    out_dir    = os.path.join(OUTPUT_DIR, f"awd_{total_iter}iter")
    os.makedirs(out_dir, exist_ok=True)

    logger.info("=" * 60)
    logger.info(f"  AWD Resume: {DONE_ITER}iter → {total_iter}iter (+{ADD_ITER})")
    logger.info(f"  이전 결과: {ORTHO_DIR}")
    logger.info(f"  출력:      {out_dir}")
    logger.info(f"  α={ALPHA} | lr={LR} | tol={TOL}")
    logger.info("=" * 60)

    # 파일 확인
    for key in KEYS:
        assert os.path.exists(phi_path(key)),   f"원본 phi 없음: {key}"
        assert os.path.exists(ortho_path(key)), f"저장된 ortho phi 없음: {key}"

    start = time.time()

    # STEP 1: phi 원본 로드
    logger.info(f"\nSTEP 1: phi {len(KEYS)}개 로드 (CPU bfloat16)...")
    phi_flats, sorted_keys, ref_shapes = [], None, None
    for key in KEYS:
        phi_dict = torch.load(phi_path(key), map_location="cpu", weights_only=True)
        if sorted_keys is None:
            sorted_keys = sorted(phi_dict.keys())
            ref_shapes  = {k: (v.shape, v.dtype) for k, v in phi_dict.items()}
        phi_flats.append(dict_to_flat(phi_dict, sorted_keys))
        del phi_dict; gc.collect()
        logger.info(f"  [{key}] 완료 | {mem_usage()}")

    total_numel = phi_flats[0].numel()

    # STEP 2: δ 복원
    logger.info(f"\nSTEP 2: δ 복원 (원본 - 저장된 ortho)...")
    delta = recover_delta(sorted_keys)

    # 복원 검증: 현재 cos 확인
    logger.info(f"\nSTEP 2-1: 복원 후 cos 확인 (21iter 완료 시점과 일치해야 함)...")
    avg_resume = measure_cosine(phi_flats, delta=delta)
    logger.info(f"  현재 평균 |cos|: {avg_resume:.6f}")
    logger.info(f"  ※ 21iter 완료 시점: 0.446815 — 일치하면 δ 복원 성공!")

    # STEP 3: 추가 최적화
    logger.info(f"\nSTEP 3: 추가 {ADD_ITER}iter 최적화 (iter {DONE_ITER+1} ~ {total_iter})...")
    prev_loss = float("inf")

    for it in range(ADD_ITER):
        t0 = time.time()
        global_it = DONE_ITER + it + 1
        logger.info(f"\n  ── iter [{global_it:4d}/{total_iter}] ──────────────────")

        L, L_O, L_R, grad = compute_awd_loss(delta, phi_flats, alpha=ALPHA)
        loss_val = float(L)
        delta -= LR * grad
        del grad; os_free()

        elapsed = time.time() - t0
        logger.info(f"  iter [{global_it:4d}/{total_iter}] ✅ "
                    f"loss={loss_val:.6f} L_O={L_O:.6f} norm_δ={L_R:.4f} | "
                    f"{elapsed:.1f}s | {mem_usage()}")

        if abs(prev_loss - loss_val) < TOL and it > 10:
            logger.info(f"  ✅ 수렴! iter={global_it}")
            break
        prev_loss = loss_val

    # STEP 4: 최종 cos 측정
    logger.info("\nSTEP 4: 직교화 후 코사인 유사도...")
    avg_before_resume = 0.446815   # 21iter 완료 시점 기록값
    avg_after = measure_cosine(phi_flats, delta=delta)
    reduction_total = (avg_resume - avg_after) / avg_resume * 100
    logger.info(f"  평균 |cos|: {avg_after:.6f}")
    logger.info(f"  추가 감소 ({DONE_ITER}→{total_iter}iter): {reduction_total:.1f}%")

    # STEP 5: 저장
    logger.info("\nSTEP 5: phi_i - δ 저장...")
    saved_paths = {}
    for i, key in enumerate(KEYS):
        ortho_flat = phi_flats[i].float() - delta.float()
        result, offset = {}, 0
        for k in sorted_keys:
            shape, dtype = ref_shapes[k]
            numel = 1
            for s in shape: numel *= s
            result[k] = ortho_flat[offset:offset+numel].reshape(shape).to(dtype).clone()
            offset += numel
        del ortho_flat; gc.collect()
        save_path = os.path.join(out_dir, f"phi_{key}.pt")
        torch.save(result, save_path)
        del result; gc.collect()
        size_mb = os.path.getsize(save_path) / 1024**2
        logger.info(f"  [{key}] → {save_path} ({size_mb:.1f} MB)")
        saved_paths[key] = save_path

    del phi_flats, delta; os_free()

    elapsed = round(time.time() - start, 1)
    manifest = {
        "method":        "AWD Resume",
        "phi_dir":       PHI_DIR,
        "ortho_dir":     ORTHO_DIR,
        "keys":          KEYS,
        "done_iter":     DONE_ITER,
        "add_iter":      ADD_ITER,
        "total_iter":    total_iter,
        "alpha":         ALPHA,
        "lr":            LR,
        "cos_at_resume": round(avg_resume, 6),
        "cos_after":     round(avg_after, 6),
        "created_at":    time.strftime("%Y-%m-%d %H:%M:%S"),
        "elapsed_sec":   elapsed,
        "files":         saved_paths,
    }
    with open(os.path.join(OUTPUT_DIR, f"manifest_awd_{total_iter}iter.json"), "w") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    logger.info(f"\n{'='*60}")
    logger.info(f"  AWD Resume 완료! 총 {total_iter}iter | 소요: {elapsed}초")
    logger.info(f"  최종 평균 |cos|: {avg_after:.6f}")
    logger.info(f"{'='*60}")
    return manifest


if __name__ == "__main__":
    run()

06:20:11 [INFO] ============================================================
06:20:11 [INFO]   AWD (Adaptive Weight Disentanglement)
06:20:11 [INFO]   Xiong et al., arXiv:2411.18729, 2024
06:20:11 [INFO]   phi 폴더  : /workspace/auto_personality/phi_vectors_llama
06:20:11 [INFO]   출력 폴더 : ./ortho_phi_vectors/awd
06:20:11 [INFO]   대상      : CON_High | NEU_High | EXT_High | AGR_High | OPN_High
06:20:11 [INFO]   α=0.1 | lr=0.01 | max_iter=300
06:20:11 [INFO]   DEVICE=cuda | phi+delta CPU, 쌍별 GPU 처리
06:20:11 [INFO] ============================================================
06:20:11 [INFO] 
STEP 1: phi 5개 로드 (CPU bfloat16 — 읽기 전용, delta/grad는 float32)...
06:20:11 [INFO]   [CON_High] 로드 중...
06:20:22 [INFO]   [CON_High] 완료 | RAM 15.5GB | VRAM 0.5/95GB
06:20:22 [INFO]   [NEU_High] 로드 중...
06:20:34 [INFO]   [NEU_High] 완료 | RAM 31.1GB | VRAM 0.5/95GB
06:20:34 [INFO]   [EXT_High] 로드 중...
06:20:46 [INFO]   [EXT_High] 완료 | RAM 46.0GB | VRAM 0.5/95GB
06:20:46 [INFO]   [AGR_High] 로드 중...
06:20:57 [I